### 1. ClinicalTrials.gov

Endpoint-base:

https://clinicaltrials.gov/api/v2/studies

Filtro proposto:

query.cond=Breast Cancer

## Principais atributos:

| Domínio       | Informações                               |
| ------------- | ----------------------------------------- |
| Identificação | Código NCT e título                       |
| Classificação | Tipo, fase e desenho do estudo            |
| Situação      | Recrutando, concluído, suspenso etc.      |
| Datas         | Início, conclusão e atualização           |
| Participantes | Quantidade prevista ou efetiva            |
| Intervenções  | Medicamentos, procedimentos e tratamentos |
| Patrocínio    | Organização responsável                   |
| Localização   | País, cidade e instituição                |
| Condições     | Doenças pesquisadas                       |

###Faz a chamada do notebook de config.

In [0]:
%run /Users/wellingtondmf@gmail.com/config

###Parametros base para ETL da Clinial Trials

In [0]:
CLINICAL_URL = "https://clinicaltrials.gov/api/v2/studies"

params = { "query.cond": "Breast Cancer",
               "format": "json",
             "pageSize": 1000,
           "countTotal": "true"}

clinical_rows = []
page_token = None
page_number = 1
total_api = None

###Processo para chamada da API e Coleta dos dados.

In [0]:
while True:
    request_params = params.copy()

    if page_token:
        request_params["pageToken"] = page_token

    response = http.get(CLINICAL_URL, params=request_params,timeout=120)
    response.raise_for_status()

    data = response.json()
    studies = data.get("studies", [])

    if total_api is None:
        total_api = data.get("totalCount")

    for study in studies:
        identification = (study.get("protocolSection", {}).get("identificationModule", {}))

        clinical_rows.append((INGESTION_ID,
                              identification.get("nctId"),
                              page_number,
                              json.dumps(study, ensure_ascii=False),
                              response.url,
                              COLLECTED_AT))

    print(f"Página {page_number}: "
          f"{len(studies)} estudos coletados")

    page_token = data.get("nextPageToken")

    if not page_token:
        break

    page_number += 1
    time.sleep(0.2)

###Estruturação do Schema da tabela.

In [0]:
clinical_schema = StructType([StructField("ingestion_id", StringType(), False),
                              StructField("nct_id", StringType(), False),
                              StructField("page_number", IntegerType(), False),
                              StructField("payload", StringType(), False),
                              StructField("source_url", StringType(), False),
                              StructField("collected_at", TimestampType(), False)])

###Cria a tabela a partir de dataframe.

In [0]:
clinical_df = spark.createDataFrame(clinical_rows, schema=clinical_schema)

clinical_df.write.format("delta")\
                 .mode("overwrite")\
                 .option("overwriteSchema", "true")\
                 .saveAsTable(f"{CATALOG}.{SCHEMA}.brz_clinical_trials")

###Valida os dados entre origem e destino.

In [0]:
clinical_count = spark.table(f"{CATALOG}.{SCHEMA}.brz_clinical_trials").count()

print(f"Quantidade informada pela API: {total_api}")
print(f"Quantidade persistida: {clinical_count}")

assert clinical_count == len(clinical_rows)
assert clinical_count == total_api

In [0]:
%sql
SELECT * 
  FROM mvp_eng_dados.mvp_cancer.brz_clinical_trials